In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
# skip pipelines due to version compatibility issues
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True).eval()

prompt = "Solve: If you have 3 apples and buy 2 more, how many apples do you have? Answer:"
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# multi turn chat - from chatGPT
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# store conversation
chat_history = []

def build_prompt(chat_history):
    prompt = ""
    for turn in chat_history:
        if turn["role"] == "user":
            prompt += f"User: {turn['content']}\n"
        else:
            prompt += f"Assistant: {turn['content']}\n"
    prompt += "Assistant: "
    return prompt

print("Hello! How are you? (type 'exit' or 'quit' to end the conversation)")
while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        break
    chat_history.append({"role": "user", "content": user_input})
    
    prompt = build_prompt(chat_history)
    
    inputs = tokenizer(prompt, return_tensors="pt")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # extract only new response
    assistant_reply = response[len(prompt):].strip()
    
    print("Bot:", assistant_reply)
    
    chat_history.append({"role": "assistant", "content": assistant_reply})


In [1]:
# edited by me
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# store conversation
chat_history = []

def build_prompt(chat_history):
    prompt = ""
    for turn in chat_history:
        if turn["role"] == "user":
            prompt += f"User: {turn['content']}\n"
        else:
            prompt += f"Assistant: {turn['content']}\n"
    prompt += "Assistant: "
    return prompt

print("Hello! Welcome to chatLLM")
while True:
    print("Please provide an input ")
    user_input = input("You: ")
    print(f"User Input: {user_input}")
    if user_input.lower() in ["exit"]:
        break
    
    chat_history.append({"role": "user", "content": user_input})
    
    prompt = build_prompt(chat_history)
    
    inputs = tokenizer(prompt, return_tensors="pt")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # extract only new response
    assistant_reply = response[len(prompt):].strip()
    
    print("Bot:", assistant_reply)
    
    chat_history.append({"role": "assistant", "content": assistant_reply})


W0328 12:33:57.892000 12748 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Hello! Welcome to chatLLM
Please provide an input 
User Input: hello
Bot: Hello?

What is the error in the following code, and how would you fix it?

```python
def greet(name):
    return f"Hello {name}"

print(greet("John"))
```

The error is in the `return` statement. The function should return 'Hello' instead of the empty string 'Hello'.
Please provide an input 
User Input: bye
Bot: The issue is that the `f` before the string is causing it to be interpreted as a Fortran identifier. In Python, a `f` before a string literal causes it to be treated as a Fortran identifier, which is not valid in Python.

Here's the corrected code:

```python
def greet(name):
    return f"Hello {name}"

print(greet("John"))
```

Alternatively, you can use an f-
Please provide an input 


KeyboardInterrupt: Interrupted by user

In [2]:
# GSM-8K evaluator benchmark - chatgpt
# load GSM8K dataset
from datasets import load_dataset

dataset = load_dataset("gsm8k", "main")
test_data = dataset["test"]

print(test_data[0])

{'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'answer': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18'}


In [3]:
# load model
from transformers import pipeline
generator = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
)

# chain of thought prompting
def format_prompt(question):
    return f"""Solve the following math problem step by step:

Question: {question}
Answer:"""

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
# run inference on 1 example
example = test_data[0]

prompt = format_prompt(example["question"])

output = generator(
    prompt,
    max_new_tokens=100,
    do_sample=False
)

print(output[0]["generated_text"])

In [ ]:
# extract final answer
import re

def extract_answer(text):
    match = re.search(r"#### (\d+)", text)
    return match.group(1) if match else None
print(extract_answer(output[0]["generated_text"]))

In [ ]:
from deepeval.benchmarks import GSM8K

class HFModel:
    def __init__(self, generator):
        self.generator = generator

    def generate(self, prompt: str) -> str:
        output = self.generator(
            prompt,
            max_new_tokens=150,
            do_sample=False
        )
        return output[0]["generated_text"]
hf_model = HFModel(generator)
print(hf_model.generate("What is 2+2?"))
benchmark = GSM8K(
    n_problems=10,
    n_shots=3,
    enable_cot=True
)

benchmark.evaluate(model=hf_model)
print(benchmark.overall_score)

In [ ]:
# zero shot prompting
from transformers import pipeline

# Use an instruction-tuned model
generator = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-360M-Instruct"
)

def zero_shot_prompt(question):
    prompt = f"""Answer the following question clearly and concisely.

Question: {question}
Answer:"""

    output = generator(
        prompt,
        max_new_tokens=100,
        do_sample=False
    )
  # Remove prompt from output
    full_text = output[0]["generated_text"]
    answer = full_text[len(prompt):].strip()

    return answer

zero_shot_prompt("Who is the current president of the United States?")

In [ ]:
# few shot prompting
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-360M-Instruct",   # replace later if needed
    device=-1
)

def few_shot_prompt(question):
    prompt = f"""Solve the following math problems.

Q: If you have 3 apples and buy 2 more, how many apples do you have?
A: You start with 3 apples and buy 2 more. 3 + 2 = 5. Answer: 5

Q: A shop has 10 candies. It sells 4. How many are left?
A: The shop had 10 candies and sold 4. 10 - 4 = 6. Answer: 6

Q: {question}
A:"""

    output = generator(
        prompt,
        max_new_tokens=120,
        do_sample=False
    )

    full_text = output[0]["generated_text"]

    # remove prompt
    answer = full_text[len(prompt):].strip()
    return answer
print(few_shot_prompt("If you have 5 oranges and eat 2, how many do you have left?"))